# AwareLiquid · Track 1A — Qwen-2.5-3B + MT adapter (Kaggle GPU)

Scale validation. Phase 5b confirmed the MT residual adapter works on Qwen-2.5-1.5B (PPL -27.7%). Now re-run on **Qwen-2.5-3B-Instruct** so needle-in-haystack has *non-zero base scores* — the 1.5 B was simply too small to retrieve a needle in 4 k context.

**Settings vs Phase 5b:**
- MODEL: `Qwen/Qwen2.5-3B-Instruct` (~6 GB fp16)
- SEQ_LEN: 384 (tighter than 1.5 B's 512 — 3 B doubles activation mem)
- STEPS=1000, BATCH=1, GRAD_ACCUM=8, MT_EVERY=4 (identical recipe)

**Hardware:** GPU T4 (16 GB) preferred; P100 (16 GB) also fits. Wall-clock target: ~5 h.

## 0 · Pin PyTorch (sm_60 + sm_75)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    'torch==2.4.1', 'torchvision==0.19.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'])
print('torch pinned to 2.4.1+cu121')
import sys as _s
if 'torch' in _s.modules:
    import os, signal; os.kill(os.getpid(), signal.SIGTERM)

## 1 · GPU sanity

In [ ]:
import torch, platform
print('python', platform.python_version()); print('torch', torch.__version__)
assert torch.cuda.is_available(), 'GPU required'
cap = torch.cuda.get_device_capability(0)
print('gpu', torch.cuda.get_device_name(0), f'sm_{cap[0]}{cap[1]}')
print('mem_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
x = torch.randn(4, 4, device='cuda'); print('cuda_smoke_ok', round((x @ x).sum().item(), 3))

## 2 · Clone repo + install deps

In [ ]:
import os, subprocess
REPO, DIR = 'https://github.com/everest-an/M1.git', '/kaggle/working/M1'
if not os.path.exists(DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, DIR])
os.chdir(DIR); subprocess.check_call(['git', 'log', '-1', '--oneline'])

In [ ]:
!pip install -q -r requirements.txt accelerate safetensors peft datasets
!python -c "from mt_lnn.llama_adapter import attach_mt_adapters; print('adapter import OK')"

## 3 · Train + PPL + Needle (Qwen-2.5-3B)

In [ ]:
%env MODEL=Qwen/Qwen2.5-3B-Instruct
%env SEQ_LEN=384
%env BATCH=1
%env GRAD_ACCUM=8
%env STEPS=1000
%env MT_EVERY=4
%env NEEDLE_CONTEXTS=1024 2048 4096
%env NEEDLE_SAMPLES=5
%env OUT_DIR=/kaggle/working/checkpoints/qwen3b_mt_adapter
%env RESULT_DIR=/kaggle/working/benchmarks/kaggle_qwen3b_run
!sed -i 's|python -m pytest tests/test_llama_adapter.py -q|echo "skipping smoke test"|' scripts/cloud_llama_mt_experiment.sh
!bash scripts/cloud_llama_mt_experiment.sh

## 4 · Package artefacts

In [ ]:
import shutil, glob
from pathlib import Path
out = Path('/kaggle/working/awareliquid_qwen3b_artifacts'); out.mkdir(parents=True, exist_ok=True)
for ckpt in sorted(glob.glob('/kaggle/working/checkpoints/qwen3b_mt_adapter/*.pt'))[-2:]:
    shutil.copy(ckpt, out / Path(ckpt).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_qwen3b_run/*.json'):
    shutil.copy(j, out / Path(j).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_qwen3b_run/*.log'):
    shutil.copy(j, out / Path(j).name)
archive = shutil.make_archive('/kaggle/working/awareliquid_qwen3b', 'zip', out)
print('archive:', archive, '·', round(Path(archive).stat().st_size / 1024**2, 1), 'MB')

## 5 · Eyeball

In [ ]:
import json, pathlib
for name in ('ppl_ablation.json', 'needle.json'):
    p = pathlib.Path('/kaggle/working/benchmarks/kaggle_qwen3b_run') / name
    if p.exists():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])